# CUAD — BERT Multi-Label Clause Classification

## Project objective

We are building a contract-review system that identifies important legal clauses in the CUAD dataset. This notebook fine-tunes **pretrained BERT** to predict which of the **41 clause categories** appear in contract text. A single window can contain more than one category.

Our long-term goal is to help reviewers find clauses that may require closer attention. **Clause detection is the first step; a separate risk assessment will be needed before we can flag potentially high-risk contracts.**

## Modeling approach

| Component | Our choice | Purpose |
|---|---|---|
| Pretrained checkpoint | `bert-base-uncased` | Starting point for learning legal-clause patterns |
| Task | Multi-label sequence classification | Predict 41 categories independently for each window |
| Input | Version 4 token windows, maximum 512 BERT tokens | Fit BERT without discarding the end of a long contract |
| Target | 41-position multi-hot `labels` vector | Represent all annotated clause categories in a window |
| Labeling | Per-Category Label Masking (PCLM) | Ignore uncertain partial-clause labels in the training loss |
| Training loss | Masked binary cross-entropy with logits | Learn positive and negative categories without penalizing ignored ones |
| Evaluation | All windows of each validation contract | Measure how well the model finds categories across complete contracts |

**Data separation:** The preprocessing notebook already split contracts into **326 training, 82 validation, and 102 test** contracts. We preserve these exact splits, select model settings on validation data, and leave the test data unused until final evaluation.

**Important distinction:** Version 4's 99%+ *annotation coverage* measures preprocessing, **not classification accuracy**. Actual model performance is measured after training.

## Running this notebook

First run a small **smoke test** (`SMOKE_TEST = True`) to check that data loading, model training, and evaluation execute correctly. Then set `SMOKE_TEST = False` and rerun all cells for a full training experiment. A smoke test is not a performance result. GPU support is helpful because BERT training is computationally expensive.

## 1. Environment and training settings

We use **PyTorch** to optimize the model and **Hugging Face Transformers** to load pretrained BERT. The classification head has 41 outputs. The `bert-base-uncased` tokenizer matches the pretrained checkpoint used in preprocessing.

`SMOKE_TEST` limits the training and validation samples for a quick pipeline check. `TRAIN_EPOCHS`, `BATCH_SIZE`, and `LEARNING_RATE` control the full training run. These settings are starting values and should be reported with the validation results.

If `torch` or `transformers` is missing, uncomment the `%pip install` line in a separate cell, run it, and restart the notebook kernel.

In [ ]:
%pip install -U torch transformers scikit-learn

In [ ]:
# Install only if needed, in a separate cell, then restart kernel:
# %pip install -U torch transformers scikit-learn

import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import f1_score, precision_score, recall_score

MODEL_NAME = "bert-base-uncased"
DATA_DIR = Path("../data/processed/tokenized_bert_v4")
MODEL_DIR = Path("../data/processed/bert_clause_model_v4")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
SMOKE_TEST = False  # Change to False after the quick test works
TRAIN_EPOCHS = 1
BATCH_SIZE = 4
LEARNING_RATE = 2e-5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print("Device:", device)
print("PyTorch:", torch.__version__)
print("Mode:", "Quick smoke test only" if SMOKE_TEST else "Full training")
if device.type == "cpu":
    print("Note: Full BERT training on CPU can be very slow. A GPU is recommended.")

## 2. Load the processed dataset

We load the **final Version 4** outputs from `data/processed/tokenized_bert_v4/`. Each training window already contains BERT token IDs, an attention mask, 41 target labels, and 41 supervision masks.

We load **every validation window** because a contract may contain relevant clauses in different locations. We also load the true contract-level labels for evaluation. The test set is not used in this notebook.

The checks below confirm the number of categories, BERT token limit, and separation between training and validation contracts.

In [ ]:
def read_jsonl(path):
    if not path.exists():
        raise FileNotFoundError(f"Run 03_CUAD_final_preprocessing.ipynb first. Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(DATA_DIR / "train.jsonl")
val_windows_all = read_jsonl(DATA_DIR / "validation_all_windows.jsonl")
val_contract_gold = read_jsonl(DATA_DIR / "validation_contract_gold.jsonl")
label2id = json.loads((DATA_DIR / "label2id.json").read_text(encoding="utf-8"))
id2label = {int(v): k for k, v in label2id.items()}
num_labels = len(label2id)
assert num_labels == 41
assert all(len(r["labels"]) == 41 and len(r["label_mask"]) == 41 for r in train_rows)
assert all(len(r["input_ids"]) <= 512 for r in train_rows + val_windows_all)
assert not ({r["contract_title"] for r in train_rows} & {r["contract_title"] for r in val_windows_all})
assert len(val_contract_gold) == 82
print("Train windows:", len(train_rows))
print("Validation windows (all):", len(val_windows_all))
print("Validation contracts:", len(val_contract_gold))
print("Labels:", num_labels)

## 3. Prepare input batches

- **`input_ids`:** Numeric IDs for the WordPiece tokens in each window.
- **`attention_mask`:** `1` for actual tokens and `0` for padded positions.
- **`labels`:** A 41-number target vector; `1` means that category has a complete annotated answer in the window.
- **`label_mask`:** A 41-number supervision vector; `0` means the corresponding category is uncertain because an answer is only partially visible.

We add padding **within each batch**, rather than padding every saved example to 512 tokens. This uses less unnecessary memory while still producing equal-length inputs within a batch.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

class WindowDataset(Dataset):
    def __init__(self, rows, training=False):
        self.rows = rows
        self.training = training
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        return self.rows[index]

def collate_windows(batch):
    max_len = max(len(r["input_ids"]) for r in batch)
    inputs = {
        "input_ids": torch.tensor([r["input_ids"] + [tokenizer.pad_token_id] * (max_len-len(r["input_ids"])) for r in batch], dtype=torch.long),
        "attention_mask": torch.tensor([r["attention_mask"] + [0] * (max_len-len(r["attention_mask"])) for r in batch], dtype=torch.long),
    }
    if all("token_type_ids" in r for r in batch):
        inputs["token_type_ids"] = torch.tensor([r["token_type_ids"]+[0]*(max_len-len(r["token_type_ids"])) for r in batch], dtype=torch.long)
    if "labels" in batch[0]:
        inputs["labels"] = torch.tensor([r["labels"] for r in batch], dtype=torch.float32)
        inputs["label_mask"] = torch.tensor([r["label_mask"] for r in batch], dtype=torch.float32)
    return inputs

selected_train = train_rows
selected_val_gold = val_contract_gold
selected_val_windows = val_windows_all
if SMOKE_TEST:
    selected_train = random.Random(SEED).sample(train_rows, min(64,len(train_rows)))
    selected_val_gold = val_contract_gold[:5]
    chosen = {r["contract_title"] for r in selected_val_gold}
    selected_val_windows = [r for r in val_windows_all if r["contract_title"] in chosen]

train_loader = DataLoader(WindowDataset(selected_train, training=True), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_windows)
val_loader = DataLoader(WindowDataset(selected_val_windows), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_windows)
print("Selected training windows:", len(selected_train))
print("Selected validation contracts:", len(selected_val_gold))
print("Selected validation windows:", len(selected_val_windows))

## 4. Fine-tune BERT using masked binary cross-entropy

BERT produces **41 logits**, one per clause category. We calculate a binary cross-entropy loss for each category separately, then multiply that loss by the corresponding `label_mask` value.

**Our training rule:** A complete annotated answer is a positive label; a category with no overlapping annotation is a negative label; a partial answer is **ignored for that category only**. The model still receives the entire text window.

The resulting *masked loss* averages only the labels we decided were safe to supervise. This is the training counterpart to our Per-Category Label Masking policy in preprocessing. We do not send `label_mask` into BERT itself; it is applied to the loss after BERT produces its predictions.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id,
)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

for epoch in range(TRAIN_EPOCHS):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader, 1):
        labels = batch.pop("labels").to(device)
        label_mask = batch.pop("label_mask").to(device)
        batch = {k: v.to(device) for k,v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        logits = model(**batch).logits
        per_label_loss = F.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        loss = (per_label_loss * label_mask).sum() / label_mask.sum().clamp_min(1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        if step == 1 or step % 25 == 0 or step == len(train_loader):
            print(f"Epoch {epoch+1} | batch {step}/{len(train_loader)} | masked loss {loss.item():.4f}")
    print(f"Epoch {epoch+1} average training loss: {total_loss / max(len(train_loader),1):.4f}")

## 5. Evaluate on validation contracts

For each validation window, we convert the model's logits into probabilities using **sigmoid**. We then take the **maximum probability for each category across all windows of the same contract**. This is our initial contract-level aggregation rule: if any window strongly indicates a category, the contract receives that score.

We compare these 41 contract-level probabilities with the known contract labels and report:

- **Precision:** Of the categories we predicted, how many were correct?
- **Recall:** Of the categories that were present, how many did we identify?
- **Micro-F1:** Overall balance of precision and recall across category predictions.
- **Macro-F1:** Average F1 across the 41 categories, highlighting weaker or less common categories.

We test several probability thresholds on **validation data only** and choose the one with the best micro-F1 in that experiment. Threshold selection is part of model development; we must not tune it using the final test set.

**A smoke test checks code execution only.** Metrics based on its small subset should not be used as evidence of model performance.

In [ ]:
model.eval()
window_probs = []
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k,v in batch.items()}
        probs = torch.sigmoid(model(**batch).logits).detach().cpu().numpy()
        window_probs.extend(probs)

by_contract = {}
for row, probs in zip(selected_val_windows, window_probs):
    title = row["contract_title"]
    if title not in by_contract:
        by_contract[title] = probs.copy()
    else:
        by_contract[title] = np.maximum(by_contract[title], probs)

contract_titles = [r["contract_title"] for r in selected_val_gold]
y_true = np.array([r["labels"] for r in selected_val_gold], dtype=int)
y_prob = np.stack([by_contract[title] for title in contract_titles])

def get_metrics(threshold):
    pred = (y_prob >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "micro_f1": float(f1_score(y_true, pred, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "precision_micro": float(precision_score(y_true, pred, average="micro", zero_division=0)),
        "recall_micro": float(recall_score(y_true, pred, average="micro", zero_division=0)),
    }

candidates = [get_metrics(t) for t in (0.2, 0.3, 0.4, 0.5, 0.6)]
for row in candidates:
    print(row)
best = max(candidates, key=lambda r:r["micro_f1"])
print("Best threshold on selected validation contracts:", best)
if SMOKE_TEST:
    print("SMOKE TEST ONLY — results on 5 contracts are NOT a reliable performance estimate.")

## 6. Save the model and experiment results

We save the BERT weights, tokenizer configuration, and a JSON training report containing the training settings, selected validation threshold, and metrics.

Files are stored under `data/processed/bert_clause_model_v4/`, with separate folders for the smoke test and full training. Model files can be large, so we do not automatically commit them to GitHub.

After verifying the smoke test, use `SMOKE_TEST = False` to train on the complete training-window dataset. If memory is limited, reduce `BATCH_SIZE` and rerun. The test set remains untouched at this stage.

In [ ]:
output = MODEL_DIR / ("smoke_test" if SMOKE_TEST else "full_train")
output.mkdir(parents=True, exist_ok=True)
model.save_pretrained(output)
tokenizer.save_pretrained(output)
report = {
    "pretrained_model": MODEL_NAME,
    "data_folder": str(DATA_DIR),
    "mode": "smoke_test_not_final" if SMOKE_TEST else "full_training",
    "training_windows": len(selected_train),
    "validation_contracts": len(selected_val_gold),
    "epochs": TRAIN_EPOCHS,
    "batch_size": BATCH_SIZE,
    "device": str(device),
    "threshold_candidates": candidates,
    "selected_threshold": best["threshold"],
    "selected_validation_metrics": best,
    "evaluation_method": "max window probabilities per contract, compared with contract-level annotations",
    "loss": "masked_binary_cross_entropy_with_logits",
}
(output / "training_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved checkpoint and report to:", output.resolve())

## 7. Results, interpretation, and next steps

### What this notebook implements

This notebook fine-tunes **pretrained `bert-base-uncased`** for **41-label clause classification** using our Version 4 tokenized windows. Per-Category Label Masking ensures that categories with partial, uncertain annotations do not contribute to the training loss. Validation predictions are combined across every window of each contract to evaluate clause detection at the contract level.

### How to interpret the results

The training loss shows how the model fits its labeled examples. **Precision, recall, micro-F1, and macro-F1 on validation contracts** show how effectively it predicts new contracts. A high preprocessing coverage percentage does not guarantee a high F1-score. Results from `SMOKE_TEST = True` are only pipeline checks and are **not final performance numbers**.

### Limitations

- Some annotations may remain incomplete inside fixed 512-token windows; their uncertain categories are masked.
- A category labeled negative means *not annotated in the window*, which is not proof that the clause is absent in every real contract.
- Taking the maximum probability across a contract's windows is a simple aggregation rule that can increase false positives; we should inspect those errors during evaluation.
- This model predicts **clause categories**, not legal risk directly.

### Next step

After completing full training, inspect category-level precision and recall, save the selected validation threshold, and evaluate **once** on the reserved test contracts. Then develop a separate, explainable risk-scoring layer using the detected clauses and relevant contract language, with human review of potentially high-risk cases.

**Presentation summary:** We prepared CUAD contracts as BERT-sized windows, used per-category masks for uncertain labels, fine-tuned a multi-label BERT classifier, and evaluated its predictions across complete validation contracts.

In [1]:
from sklearn.metrics import precision_recall_fscore_support

predictions = (y_prob >= best["threshold"]).astype(int)

p, r, f1, support = precision_recall_fscore_support(
    y_true,
    predictions,
    average=None,
    zero_division=0
)

results = []

for i in range(num_labels):
    results.append({
        "category": id2label[i],
        "support": int(support[i]),
        "precision": round(float(p[i]), 3),
        "recall": round(float(r[i]), 3),
        "f1": round(float(f1[i]), 3)
    })

results = sorted(results, key=lambda x: x["f1"])

print("10 categories with lowest F1:")
for row in results[:10]:
    print(row)

# Save results without retraining
output = MODEL_DIR / "full_train"

with open(output / "validation_by_category.json", "w") as file:
    json.dump(results, file, indent=2)

np.savez_compressed(
    output / "validation_predictions.npz",
    y_true=y_true,
    y_prob=y_prob,
    contract_titles=np.array(contract_titles)
)

print("Validation details saved!")

NameError: name 'y_prob' is not defined

In [2]:
for name in ["y_true", "y_prob", "best", "id2label"]:
    print(name, "Available" if name in globals() else "Not available")

y_true Not available
y_prob Not available
best Not available
id2label Not available
